<a href="https://colab.research.google.com/github/gadekarvishal08-cloud/IN226005202_NLP/blob/main/NLP_Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment 5
## Token Classification: POS Tagging & Chunking using DistilBERT

### Objective
To fine-tune a transformer model (DistilBERT) for:
- Part-of-Speech (POS) Tagging
- Chunking (Phrase Detection)

### Dataset Used
CoNLL-2000 Dataset

### Tools
- Python
- Hugging Face Transformers
- PyTorch
- Google Colab

In [ ]:
!pip install transformers datasets seqeval evaluate accelerate nltk -q

### Task 1: Dataset Selection

We use the CoNLL-2000 dataset which contains:
- Tokens (words)
- POS Tags (e.g., NN, VB)
- Chunk Tags (e.g., B-NP, I-NP)

POS → Word-level classification  
Chunking → Phrase-level classification

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import nltk
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

nltk.download('conll2000')

### Label Categories

#### POS Tags (44 labels)
Examples:
- NN → Noun
- VB → Verb
- JJ → Adjective
- IN → Preposition

#### Chunk Tags (23 labels)
BIO Format:
- B-NP → Beginning of Noun Phrase
- I-NP → Inside Noun Phrase
- O → Outside

### Task 2: Data Preprocessing

Steps:
1. Tokenize using DistilBERT tokenizer
2. Align labels with tokens
3. Handle:
   - Subwords → assign -100
   - Special tokens → ignore

In [ ]:
from nltk.corpus import conll2000

def parse_conll2000(tagged_sents):
    records = []
    for sent in tagged_sents:
        tokens, pos_tags, chunk_tags = [], [], []
        for word, pos, chunk in sent:
            tokens.append(word)
            pos_tags.append(pos)
            chunk_tags.append(chunk)
        records.append({
            'tokens': tokens,
            'pos_tags': pos_tags,
            'chunk_tags': chunk_tags
        })
    return records

train_sents = conll2000.iob_sents('train.txt')
test_sents = conll2000.iob_sents('test.txt')

train_records = parse_conll2000(train_sents)
test_records = parse_conll2000(test_sents)

# Split train → train + validation
split_idx = int(len(train_records) * 0.9)
val_records = train_records[split_idx:]
train_records = train_records[:split_idx]

raw_dataset = DatasetDict({
    'train': Dataset.from_list(train_records),
    'validation': Dataset.from_list(val_records),
    'test': Dataset.from_list(test_records)
})

print(raw_dataset)

### Task 3: Model Setup

We use:
- AutoModelForTokenClassification
- DistilBERT (lightweight version of BERT)

Why DistilBERT?
- Faster training
- Less memory
- Comparable performance

In [ ]:
all_splits = ['train', 'validation', 'test']

pos_labels = sorted(set(
    tag for split in all_splits
    for ex in raw_dataset[split]
    for tag in ex['pos_tags']
))

chunk_labels = sorted(set(
    tag for split in all_splits
    for ex in raw_dataset[split]
    for tag in ex['chunk_tags']
))

def create_label_map(labels):
    return {label: i for i, label in enumerate(labels)}

pos_label2id = create_label_map(pos_labels)
chunk_label2id = create_label_map(chunk_labels)

pos_id2label = {i: l for l, i in pos_label2id.items()}
chunk_id2label = {i: l for l, i in chunk_label2id.items()}

print("POS Labels:", len(pos_labels))
print("Chunk Labels:", len(chunk_labels))

### Label Categories

#### POS Tags (44 labels)
Examples:
- NN → Noun
- VB → Verb
- JJ → Adjective
- IN → Preposition

#### Chunk Tags (23 labels)
BIO Format:
- B-NP → Beginning of Noun Phrase
- I-NP → Inside Noun Phrase
- O → Outside

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

### Task 2: Data Preprocessing

Steps:
1. Tokenize using DistilBERT tokenizer
2. Align labels with tokens
3. Handle:
   - Subwords → assign -100
   - Special tokens → ignore

In [ ]:
def tokenize_and_align_labels(examples, label_key, label2id):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples[label_key]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized

### Task 3: Model Setup

We use:
- AutoModelForTokenClassification
- DistilBERT (lightweight version of BERT)

Why DistilBERT?
- Faster training
- Less memory
- Comparable performance

In [ ]:
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_labels),
    id2label=pos_id2label,
    label2id=pos_label2id
)

chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_labels),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

### Task 4: Training

Hyperparameters:
- Learning Rate: 2e-5
- Epochs: 3
- Batch Size: 16

Trainer API handles:
- Training loop
- Evaluation
- Checkpoint saving

In [ ]:
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_labels),
    id2label=pos_id2label,
    label2id=pos_label2id
)

chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_labels),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

### Task 5: Evaluation

Metric Used: seqeval

Measures:
- Precision
- Recall
- F1 Score
- Accuracy

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

metric = evaluate.load("seqeval")

def compute_metrics(label_list):
    def fn(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_preds = [
            [label_list[p] for p, l in zip(pred, lab) if l != -100]
            for pred, lab in zip(predictions, labels)
        ]

        true_labels = [
            [label_list[l] for l in lab if l != -100]
            for lab in labels
        ]

        results = metric.compute(predictions=true_preds, references=true_labels)

        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]
        }

    return fn

### Task 6: Inference

We test the model on custom sentences.

In [ ]:
def get_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        report_to="none"
    )

### Task 7: Comparison

| Feature        | POS Tagging | Chunking |
|---------------|------------|----------|
| Level         | Word       | Phrase   |
| Complexity    | Easy       | Medium   |
| Labels        | POS Tags   | BIO Tags |
| Dependency    | Independent| Context-based |

Conclusion:
- POS is easier
- Chunking requires understanding context

In [ ]:
def get_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",      # ✅ FIXED HERE
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        report_to="none"
    )

In [ ]:
print("POS Evaluation")
pos_results = pos_trainer.evaluate(pos_tokenized["test"])
print(pos_results)

print("\nChunk Evaluation")
chunk_results = chunk_trainer.evaluate(chunk_tokenized["test"])
print(chunk_results)

In [ ]:
def predict(sentence, model, tokenizer, id2label):
    words = sentence.split()

    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()
    word_ids = inputs.word_ids(batch_index=0)

    result = []
    seen = set()

    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)
        result.append((words[word_id], id2label[predictions[i]]))

    return result

In [ ]:
sentence = "John works at Google in California"

print("\nPOS Tags:")
print(predict(sentence, pos_model, tokenizer, pos_id2label))

print("\nChunk Tags:")
print(predict(sentence, chunk_model, tokenizer, chunk_id2label))

📌 Report: POS Tagging vs Chunking
What is POS Tagging?

POS tagging assigns grammatical categories to each word such as noun, verb, adjective, etc.

Example:
Sentence: John works at Google
POS: NNP VBZ IN NNP

What is Chunking?

Chunking groups words into phrases using BIO tagging.

Example:

B-NP → Start of noun phrase
I-NP → Inside noun phrase
Key Difference
POS → Word-level classification
Chunking → Phrase-level grouping
Challenges Faced
Subword tokenization (WordPiece issue)
Label alignment with tokens
Handling -100 for ignored tokens
BIO tagging complexity
Class imbalance (O tag dominant)
Observations
DistilBERT performs very well with fewer parameters
POS tagging is easier and more accurate
Chunking requires context understanding
Transformer models capture syntax effectively
Conclusion

Fine-tuning transformer models for token classification is efficient and accurate. Proper label alignment is the most critical step.